# HLA Motif UMAP Pipeline
**Summary:** Generates UMAP embeddings and performs HDBSCAN clustering to analyze HLA motifs and modifications.
Extracts position-specific enrichment and plots scatter representations.

**Required Files:**
- Numpy Embeddings (`../seq_emb_con_int.npy`, etc.)
- HLA Rescore Results


In [ ]:
import random
import numpy as np
import os
import tensorflow as tf
from Bio import motifs
from Bio.Seq import Seq
import logomaker
import matplotlib.pyplot as plt
import pandas as pd
from spectrum_fundamentals.mod_string import internal_without_mods
import re
from collections import Counter
from glob import iglob
from pathlib import Path
import matplotlib
import warnings
import umap.umap_ as umap
from umap.parametric_umap import ParametricUMAP
import seaborn as sns
from adjustText import adjust_text
from sklearn.cluster import AgglomerativeClustering
import hdbscan
from tqdm import tqdm
import itertools
import math
from sklearn.decomposition import PCA
from random import shuffle
import ast
import scipy.stats as stats
from sklearn.preprocessing import MinMaxScaler
from matplotlib import pyplot as plt
from joblib import Parallel, delayed


## Configuration
Paths and variables needed to run this notebook.


In [ ]:
SEQ_EMBEDDING_CONCAT_PATH = '<PATH_TO_SEQ_EMBEDDING_CONCAT>'
SEQ_EMBEDDING_MUL_PATH = '<PATH_TO_SEQ_EMBEDDING_MUL>'
LS_EMB_PATH = '<PATH_TO_LS_EMB>'
ATTENTION_EMB_PATH = '<PATH_TO_ATTENTION_EMB>'
ALLELES_GLOB = '<PATH>'
CLUSTER_RESULTS_GLOB = '<PATH>'
OUTPUT_DIR = '<PATH_TO_OUTPUT>'

random.seed(42) 
np.random.seed(42)
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
warnings.filterwarnings('ignore')
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42


In [ ]:
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
sequences_to_check=['P_35','L','Q_34','N_7','K','R','S','T','Y','A','C','D','E','F','G','Q','N','I','V','M','H','P','W','Q_7',
                    'S_21','Y_21','T_21','K_1','D_34','R_34','K_34','E_34','L_34','K_35','N_34','H_35','I_34','W_35','C_34','C_35',
                    'H_34','K_36','R_36','K_121','K_535','K_1293','K_1990','C_312','K_12118','K_129317','K_19903','K_1263']

# %%
seq_embedding_concat = np.load(SEQ_EMBEDDING_CONCAT_PATH)
seq_embedding_mul = np.load(SEQ_EMBEDDING_MUL_PATH)
ls_emb = np.load(LS_EMB_PATH)
attention_emb = np.load(ATTENTION_EMB_PATH)

# %%
all_aleles = [f for f in iglob(ALLELES_GLOB)]

def plot_umap(ls_emb_train,ls_emb_fil,base_AA_label,fil_seq,train_label,cluster_exp_label,path,distances,clusters_dict,keys,cluster_wrong_total):
    count_successful_clusters =0
    most_clusters = 0
    default_value = 0
    combinations = itertools.product(np.arange(1,4,0.15), np.arange(1,2.55,0.25), np.arange(1,3,1))
    found_keys_dict = {'{}_{}_{}'.format(round(a,2), round(b,2),c): default_value for a, b,c in combinations}
    for n_neigbhours in np.arange(1,3,1):
        for dist in np.arange(1,4,0.15):
                for spread_in in np.arange(1,2.55,0.5):
                    dist = round(dist,2)
                    spread_in = round(spread_in,2)
                    if (len(base_AA_label)-n_neigbhours)<2:
                        continue
                    umap_model = ParametricUMAP(n_components=3,n_neighbors=len(base_AA_label)-n_neigbhours,min_dist=dist,spread=dist+spread_in, random_state=1,n_jobs=1,metric='cosine',autoencoder_loss=True)

                    _ = umap_model.fit(np.asarray(ls_emb_train).reshape(len(base_AA_label),(32*512)),y=train_label)

                    umap_embeddings =  umap_model.transform(np.asarray(ls_emb_fil).reshape(len(fil_seq),(32*512)))#len(fil_seq)
                    if np.isnan(umap_embeddings).any():
                        continue
                    embedding_df = pd.DataFrame(umap_embeddings, columns=['UMAP 1', 'UMAP 2', 'UMAP 3'])

                    clusterer = hdbscan.HDBSCAN(min_cluster_size=2)
                    clusters = clusterer.fit_predict(umap_embeddings)
                    
                    n_clusters = len(set(clusters))
                    # You can adjust this
                    clustering = AgglomerativeClustering(n_clusters=n_clusters)
                    clusters = clustering.fit_predict(umap_embeddings)
                    cluster_correctly = False
                    cluster_correctly_total = 0
                    cluster_wrong_total_int = 0

                    clustered_aa_total = 0
                    embedding_df['label'] = fil_seq
                    embedding_df['clusters'] = clusters
                    
                    clustering = AgglomerativeClustering(n_clusters=n_clusters)
                    clusters = clustering.fit_predict(umap_embeddings)
                    
                    embedding_df['label'] = fil_seq
                    embedding_df['clusters'] = clusters
                    plt.figure(figsize=(20, 8))
                    plt.scatter(umap_embeddings[:, 0], umap_embeddings[:, 1], c=clusters, cmap='tab20')
                    # Add labels and adjust text positions
                    texts = [
                        plt.text(embedding_df['UMAP 1'][i], embedding_df['UMAP 2'][i], str(embedding_df['label'][i]),
                                fontdict={'size': 8, 'color': 'black'})
                        for i in range(embedding_df.shape[0])
                    ]

                    # Automatically adjust text positions
                    adjust_text(texts, arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))
                    plt.title('UMAP clustering')
                    plt.xlabel('UMAP_1')
                    plt.ylabel('UMAP_2')
                    plt.colorbar(label='Cluster')
                    plt.savefig(path + '/umap_dist_' + str(dist)  + '_spread_' + str(spread_in) + '_total_neigbhors' +str(n_neigbhours) + '_clusters_' + str(most_clusters)+ '.pdf',bbox_inches='tight')
                    plt.close()
    return


# %%
background_probs = {'L': 0.09963102872087613,
 'S': 0.08331084793830007,
 'E': 0.07109697459009827,
 'A': 0.07010041485553836,
 'G': 0.06570329045649569,
 'P': 0.0631396919219813,
 'V': 0.059638206480571915,
 'K': 0.05736173590060331,
 'R': 0.05636210199094119,
 'T': 0.053526834210966406,
 'Q': 0.047697495548594454,
 'D': 0.047384808023913226,
 'I': 0.04337748902782989,
 'F': 0.03648351961077781,
 'N': 0.035878961118535974,
 'Y': 0.026620951047889675,
 'H': 0.026213139762188843,
 'C': 0.022993160926566912,
 'M': 0.021327836356916298,
 'W': 0.0121483495017377}

# %%
def get_top_3_keys(dictionary):
    return [k for k, v in sorted(dictionary.items(), key=lambda item: item[1], reverse=True)[:5]]

 # Calculate information content
def calculate_ic(prob_df, background_probs):
    ic = prob_df * np.log2(prob_df/(background_probs))
    return ic.fillna(0)

def count_matrix(sequences):
    length = len(sequences[0])
    counts = {aa: [0] * length for aa in 'ACDEFGHIKLMNPQRSTVWY'}
    for seq in sequences:
        for i, aa in enumerate(seq):
            counts[aa][i] += 1
    return pd.DataFrame(counts)

def count_matrix_mod(sequences):
    length = len(sequences[0])
    counts = {aa: [0] * length for aa in 'ACDEFGHIKLMNPQRSTVWY'.lower()}
    for seq in sequences:
        for i, aa in enumerate(seq):
            counts[aa][i] += 1
    return pd.DataFrame(counts)

def calculate_background_frequencies(sequences):
    total_counts = Counter()
    total_length = 0
    for seq in sequences:
        total_counts.update(seq)
        total_length += len(seq)
    frequencies = {aa: count / total_length for aa, count in total_counts.items()}
    return frequencies


def check_enriched_pos(ic_df):
    totals = ic_df[ic_df>0].sum(axis=1)
    most_enriched_pos = get_top_3_keys(totals)
    return most_enriched_pos


# %%
def get_top_n_position_pairs(sequences, position, n):
    # Regular expression to match the amino acid with its modification at the specified position
    pattern = r'^' + r'[A-Z](?:\[UNIMOD:\d+\])?'*(position-1) + r'([A-Z])(\[UNIMOD:\d+\])?'
    
    # Extract the amino acid and its modification (if present) at the specified position
    position_pairs = []
    for seq in sequences:
        match = re.match(pattern, seq)
        if match:
            amino_acid = match.group(1)
            modification = match.group(2) if match.group(2) else ''
            position_pairs.append((amino_acid, modification))
    
    # Count occurrences
    pair_counts = Counter(position_pairs)
    
    # Get the top N most common
    top_n = pair_counts.most_common(n)
    
    return top_n


# %%
def find_equal_and_unique_columns(df):
    # Convert DataFrame to a 2D numpy array for faster comparison
    arr = df.values.T
    
    # Create a boolean matrix of column equality
    equality_matrix = (arr[:, None] == arr).all(axis=2)
    
    # Set diagonal to False to avoid self-comparison
    np.fill_diagonal(equality_matrix, False)
    
    # Find groups of equal columns and unique columns
    equal_groups = []
    processed_columns = set()
    
    for i in range(len(equality_matrix)):
        if i not in processed_columns:
            if equality_matrix[i].any():
                group = [df.columns[i]] + list(df.columns[equality_matrix[i]])
                equal_groups.append(group)
                processed_columns.update(group)
            else:
                equal_groups.append([df.columns[i]])
                processed_columns.add(df.columns[i])
    
    return equal_groups

# %%

def find_equal_columns(df):
    # Convert DataFrame to a 2D numpy array for faster comparison
    arr = df.values.T
    
    # Create a boolean matrix of column equality
    equality_matrix = (arr[:, None] == arr).all(axis=2)
    
    # Set diagonal to False to avoid self-comparison
    np.fill_diagonal(equality_matrix, False)
    
    # Find groups of equal columns
    equal_groups = []
    for i in range(len(equality_matrix)):
        if equality_matrix[i].any():
            group = [df.columns[i]] + list(df.columns[equality_matrix[i]])
            if set(group) not in [set(g) for g in equal_groups]:
                equal_groups.append(group)
    
    return equal_groups


# %%
def adjust_base_labels_umap(ic_df,most_enriched_positions,base_AA_label, binding_pos):
    ic_df_round = ic_df.round(2)
    binding_aa = []
    c_term_aa = []
    next_label = 2
    all_aa = []
    for pos in most_enriched_positions:
        aas = ic_df_round.loc[pos,ic_df_round.loc[pos]>0.06].index.tolist()
        aas_bind = ic_df_round.loc[pos,ic_df_round.loc[pos]>0.1].index.tolist()

        if pos == binding_pos:
            binding_aa+=aas_bind
        elif pos == 8:
            c_term_aa+=aas_bind
        if len(aas) == 1:
            continue
        all_aa +=aas
    all_aa = list(set(all_aa))
    df_all = pd.DataFrame(columns=['position'] + all_aa)
    default_value = 0

    for pos in most_enriched_positions:
        new_row = dict.fromkeys(all_aa,default_value)
        new_row['position'] = pos
        aas = ic_df_round.loc[pos,ic_df_round.loc[pos]>0.06].index.tolist()
        if len(aas) == 1:
            base_AA_label[aas[0]] = next_label
            next_label+=1 
            continue
        for aa in aas:
            new_row[aa]=1
        df_all.loc[len(df_all)] = new_row
    df_all.drop(columns=['position'],inplace=True)

    results = find_equal_columns(df_all)
    flatten_results =  [item for sublist in results for item in sublist]
    total_aa = 0
    total_columns = len(df_all.columns)
    while total_aa<total_columns-1:
        total_aa+= len(flatten_results)    
        for group in results:
            existing_aas = set(group).intersection(set(base_AA_label.keys()))
            if len(existing_aas)>0:
                existing_aa = existing_aas.pop()
                for aa in group:
                    base_AA_label[aa] = base_AA_label[existing_aa]
            else:
                for aa in group:
                    base_AA_label[aa] = next_label
                next_label+=1
        row_indices = df_all[df_all[flatten_results].eq(1).any(axis=1)].index
        df_all.loc[row_indices] =0
        df_all.drop(columns=flatten_results,inplace=True)
        results = find_equal_columns(df_all)
        flatten_results =  [item for sublist in results for item in sublist]
        if len(flatten_results)==0:
            break
    for aa in df_all.columns:
        base_AA_label[aa] = next_label
        
    return base_AA_label,binding_aa,c_term_aa

# %%
def get_possible_aa_and_cluster_helper(ic_mod_df,index,aa_in_cluster,min_enriched,df_psms_mod,aa_fil,mod_enriched):
    new_seqs = []
    possible_cluster = {}
    aas = ic_mod_df.loc[index,ic_mod_df.loc[index]>min_enriched].index.tolist()
    position = index+1  # Position to analyze (1-based index)
    n = 50
    top_n_results = get_top_n_position_pairs(df_psms_mod['peptide'].values, position, n)
    pattern = r'\[UNIMOD:(\d+)\]'
    for (amino_acid, modification), count in top_n_results:
        if modification:
            if amino_acid in aas and amino_acid==aa_fil and count>=1:
                match = re.search(pattern, modification)
                new_seqs.append(amino_acid+ '_' + match.group(1))
                possible_cluster[amino_acid+ '_' + match.group(1)] = aa_in_cluster
                mod_enriched[amino_acid+ '_' + match.group(1)] = ic_mod_df.loc[index][amino_acid]
    return new_seqs,possible_cluster,mod_enriched

def get_possible_aa_and_cluster(ic_mod_df,max_sum_index,binding_pos,binding_aa,c_term_aa,df_psms_mod,aa_fil):
    new_seqs = []
    possible_cluster = {}
    mod_enriched = {}
    min_enriched = ic_mod_df.loc[max_sum_index].max()/3

    if binding_pos == max_sum_index:
        pos_new_seq,possible_cluster_pos,mod_enriched = get_possible_aa_and_cluster_helper(ic_mod_df,binding_pos,binding_aa+c_term_aa,min_enriched,df_psms_mod,aa_fil,mod_enriched)
        new_seqs+=pos_new_seq
        possible_cluster.update(possible_cluster_pos)

    if max_sum_index == 8:
        pos_new_seq,possible_cluster_pos,mod_enriched = get_possible_aa_and_cluster_helper(ic_mod_df,8,binding_aa+c_term_aa,min_enriched,df_psms_mod,aa_fil,mod_enriched)
        new_seqs+=pos_new_seq
        possible_cluster.update(possible_cluster_pos)

    return new_seqs,possible_cluster,mod_enriched


# %%
def align_peptides(peptides):
    """
    Align peptides to a length of 9, accounting for modifications in the format [UNIMOD:XX].
    - For length 8: Repeat position 7 (including modification) twice.
    - For length 10: Drop the amino acid (including modification) at position 9.
    - For length 11: Drop the amino acids (including modifications) at positions 9 and 10.
    """
    aligned_peptides = []
    
    def split_peptide(peptide):
        """
        Split a peptide sequence into a list of positions, where each position is either:
        - A single amino acid (e.g., 'A')
        - An amino acid with a modification (e.g., 'A[UNIMOD:21]')
        """
        # Match single amino acids or amino acids followed by [UNIMOD:XX]
        return re.findall(r'[A-Z](?:\[UNIMOD:\d+\])?', peptide)
    
    def join_peptide(positions):
        """
        Join a list of positions back into a peptide sequence.
        """
        return ''.join(positions)
    
    for peptide in peptides:
        # Split peptide into positions
        positions = split_peptide(peptide)
        length = len(positions)
        if length == 8:
            # Repeat position 7 (index 6) twice
            aligned_positions = positions[:6] + [positions[6], positions[6]] + [positions[7]]
        elif length == 10:
            # Drop amino acid at position 9 (index 8)
            aligned_positions = positions[:8] + [positions[9]]
        elif length == 11:
            # Drop amino acids at positions 9 and 10 (indices 8 and 9)
            aligned_positions = positions[:8] + [positions[10]]
        elif length == 9:
            # Keep as is
            aligned_positions = positions
        else:
            raise ValueError(f"Peptide length {length} is not supported.")
        
        # Join the aligned positions back into a peptide sequence
        aligned_peptide = join_peptide(aligned_positions)
        aligned_peptides.append(aligned_peptide)
    
    return aligned_peptides




def extract_aa_mod(peptide):
    return re.findall(r'([A-Z](?:\[UNIMOD:\d+\])?)', peptide)

def write_to_disk(allele_name, distances,clusters,keys,found_keys_dict,cluster_wrong_total,clustered_keys_allele,aleles,
                  no_of_succ_clusters,enriched_mods,enrich_mod_lv_all,number_mod_peptide_all,base_AA_label):
    import json
    data = {
    'distances': distances,
    'clusters': clusters,
    'keys': keys,
    'clustered_keys': found_keys_dict,
    'wrong_keys': cluster_wrong_total,
    }
    df_clusters = pd.DataFrame(data)

    df_clusters.to_csv('<PATH>'+ allele_name +'.csv')

    with open('<PATH>'+ allele_name +'.json', 'w') as f2:
        json.dump(clustered_keys_allele, f2)

    df_clusters[df_clusters['distances']==4].sort_values(['keys','clusters'],ascending=False).head(20)


    df_succ_clustered = pd.DataFrame()
    df_succ_clustered['alleles'] = aleles
    df_succ_clustered['no_of_success'] = no_of_succ_clusters
    df_succ_clustered['enriched_mods'] = enriched_mods
    df_succ_clustered['enrich_lv'] = enrich_mod_lv_all
    df_succ_clustered['no_mod_pepties'] = number_mod_peptide_all
    df_succ_clustered['length'] = df_succ_clustered['enriched_mods'].apply(lambda x: len(x))
    df_succ_clustered.to_csv('<PATH>'+ allele_name +'.csv')
    with open('<PATH>'+ allele_name +'.json', 'w') as f2:
        json.dump(base_AA_label, f2)




In [ ]:
def get_motif_umap(f):

    allele_name = f.split('/')[7].split('_')[0]
    scientific_allele_name = allele_name[0] + '*' + allele_name[1:3] + ':' + allele_name[3:5]
    #if 'A0101' not in f:
    #    return
    aleles = []
    no_of_succ_clusters = []
    enriched_mods = []
    clustered_aas = []
    enrich_mod_lv_all = []
    number_mod_peptide_all = []
    default_value = 0
    combinations = itertools.product(np.arange(1,4,0.15), np.arange(1,3,0.5), np.arange(1,3,1))
    distances = {'{}_{}_{}'.format(round(a,2), round(b,2),c): default_value for a, b,c in combinations}
    clusters = distances.copy()
    keys = distances.copy()
    cluster_wrong_total = distances.copy()
    clustered_keys_allele = {}
    clustered_keys_all_alleles = []
   

    df_psms = pd.read_csv(f,sep='\t')
    aleles.append(allele_name)
    dir_path = '<PATH>'+allele_name
    alele_path = Path(dir_path)
    alele_path.mkdir(parents=True, exist_ok=True)

    df_psms  = df_psms[df_psms['q-value']<=0.05]
    
    df_psms['peptide'] = df_psms['peptide'].apply(lambda x: x.replace('C[UNIMOD:4]','C'))
    df_psms['peptide'] = df_psms['peptide'].apply(lambda x: x.replace('[UNIMOD:1]-',''))
    df_psms['peptide'] = df_psms['peptide'].apply(lambda x: x.replace('M[UNIMOD:35]','M'))
    df_psms['peptide'] = df_psms['peptide'].apply(lambda x: x.replace('R[UNIMOD:7]','R'))
    df_psms['peptide'] = df_psms['peptide'].apply(lambda x: x.replace('_','').replace('.',''))
    df_psms['peptide'] = align_peptides(df_psms['peptide'].values)
    df_psms['unmod_peptide'] = internal_without_mods(df_psms['peptide'].values)
    df_psms['peptide_length'] = df_psms['unmod_peptide'].apply(lambda x: len(x))

    df_psms_unmod = df_psms[~df_psms['peptide'].str.contains('UNIMOD')]
    
        
    df_psms = df_psms[df_psms['peptide'].str.contains('UNIMOD')]
    df_psms['peptide'] = align_peptides(df_psms['peptide'].values)
    df_psms['aa_mod_combinations'] = df_psms['peptide'].apply(extract_aa_mod)
    
    # Flatten the list of combinations
    all_combinations = [item for sublist in df_psms['aa_mod_combinations'] for item in sublist]

    # Count unique combinations
    unique_counts = pd.DataFrame(pd.Series(all_combinations).value_counts())
    possible_mod_aa = unique_counts[(unique_counts.index.str.contains('UNIMOD')) & (unique_counts[0]>=10)].index.to_list()
    cluster_exp_labels = {}
    all_mods_enriched = {}
    number_mod_peptide = {}
    
    
    df_psms_unmod.drop_duplicates('unmod_peptide',inplace=True)
    df_psms.drop_duplicates('peptide',inplace=True)
    background_sequences = []

    for pep in df_psms_unmod['unmod_peptide'].values:
        background_sequences.append(pep)
        
    df = count_matrix(background_sequences)

    prob_df = logomaker.transform_matrix(df,normalize_values=True)
    prob_df[prob_df < 0.002] = 0
    ic_df = calculate_ic(prob_df, background_probs)
        
    most_enriched_positions = check_enriched_pos(ic_df)
    binding_pos = 0
    for position in most_enriched_positions:
        if position!=8:
            binding_pos = position
            break
    base_AA_label = {
    
    }
    base_AA_label,binding_aa,c_term_aa = adjust_base_labels_umap(ic_df,most_enriched_positions,base_AA_label,binding_pos)

    logo = logomaker.Logo(ic_df, font_name='DejaVu Sans Mono',figsize=(6, 4),fade_below=0.7)

    # Customize the plot
    logo.style_spines(visible=False)
    logo.style_spines(spines=['left', 'bottom'], visible=True)
    logo.style_xticks(rotation=0, fmt='%d', anchor=0)
    # Set y-axis label and limit
    plt.ylabel("Conservation of Amino Acids in Bits")

    plt.ylim(0,ic_df[ic_df>0].sum(axis=1).max())  # Adjust as needed based on your data
    plt.title(scientific_allele_name + ' Unmodified Peptides')
    plt.xticks([0,1,2,3,4,5,6,7,8], ['1', '2', '3', '4', '5', '6', '7', 'C-1', 'C'])
    plt.savefig(dir_path + '/motif_unmodified.pdf',bbox_inches='tight')
    plt.close()
    fil_seq = ['L', 'S', 'E', 'A', 'G', 'P', 'V', 'K', 'R', 'T', 'Q', 'D', 'I', 'F', 'N', 'Y', 'H', 'C', 'M', 'W',
                'Q_7','N_7']
    

    label_fil = []
    ls_emb_fil = []

    fil_seq = sorted(fil_seq)
    base_AA_label = dict(sorted(base_AA_label.items()))
 
    for seq in fil_seq:
        
        index = sequences_to_check.index(seq)
        label_fil.append(seq)
        ls_emb_fil.append(ls_emb[index])

    train_label = []
    ls_emb_train = []
    for seq in base_AA_label:
        index = sequences_to_check.index(seq)
        train_label.append(base_AA_label[seq])
        ls_emb_train.append(ls_emb[index])
    plot_umap(ls_emb_train,ls_emb,base_AA_label,sequences_to_check,train_label,cluster_exp_labels,dir_path,distances,clusters,keys,cluster_wrong_total)

## RUN code on all alleles to get unmodified motif and plot umaps

In [ ]:
Parallel(n_jobs=11)(delayed(get_motif_umap)(f) for f in tqdm(all_aleles))